# 1: Install libraries

In [ ]:
!pip install -q transformers datasets evaluate accelerate sentencepiece sacrebleu rouge_score bert_score nltk pandas scikit-learn openpyxl

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 974.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 4.3 MB/s eta 0:00:00


# 2: Import libraries

In [ ]:
import os
import re
import gc
import torch
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback
)

import evaluate
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

nltk.download("wordnet")
nltk.download("omw-1.4")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


GPU available: True
GPU: Tesla T4


# 3: Upload dataset

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_path = list(uploaded.keys())[0]

print("Uploaded file:", csv_path)

Saving BanglaCounter_spacing_latin_fixed.csv to BanglaCounter_spacing_latin_fixed.csv
Uploaded file: BanglaCounter_spacing_latin_fixed.csv


# 4: Load dataset

In [ ]:
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Dataset shape: (3011, 13)
Columns: ['id', 'offensive_text', 'offensive_text_en', 'counterspeech_text', 'counterspeech_text_en', 'counterspeech_type_annotator_1', 'counterspeech_type_annotator_2', 'counterspeech_type_annotator_3', 'counterspeech_type_final', 'text_type_annotator_1', 'text_type_annotator_2', 'text_type_annotator_3', 'text_type_final']


,id,offensive_text,offensive_text_en,counterspeech_text,counterspeech_text_en,counterspeech_type_annotator_1,counterspeech_type_annotator_2,counterspeech_type_annotator_3,counterspeech_type_final,text_type_annotator_1,text_type_annotator_2,text_type_annotator_3,text_type_final
0,1,"কি মনে করছ, মেয়েরা কিছু বোঝে ?","What do you think, do girls understand anything?",মেয়েদের বুদ্ধি নিয়ে হেয় করা যুক্তিসঙ্গত নয়; অন...,Belittling women's intelligence is unreasonabl...,fact_based,fact_based,fact_based,fact_based,gender_abuse,gender_abuse,gender_abuse,gender_abuse
1,2,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ","You're a whore, roaming the streets turning tr...","ভাই, মত না মিললেও নারীর ব্যক্তিগত জীবন নিয়ে সম...","Look, keep respect when disagreeing about a wo...",moral_appeal,moral_appeal,moral_appeal,moral_appeal,threat,gender_abuse,gender_abuse,gender_abuse
2,3,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,It's the Hindus who are smuggling the country'...,"দেখেন, হিন্দুদের সম্পর্কে ধারণা নিয়ে চাপ দিলে ...","Look, pressure around assumptions about Hindus...",warning_of_consequences,warning_of_consequences,warning_of_consequences,warning_of_consequences,religious_hate,religious_hate,religious_hate,religious_hate
3,4,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,As if a woman could ever drive a car!,"ভাই, মেয়েদের গাড়ি চালানো নিয়ে অসম্মান নয়, নিজে...","Look, state your point about women driving wit...",fact_based,moral_appeal,moral_appeal,moral_appeal,gender_abuse,gender_abuse,gender_abuse,gender_abuse
4,5,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,Listening to this goat's words makes my blood ...,কারও বুদ্ধি টেনে অপমান না করে বক্তব্য দিন; সম্...,State your point without insulting over someon...,moral_appeal,discouraging_abuse,moral_appeal,moral_appeal,abusive_language,abusive_language,abusive_language,abusive_language


# 5: Keep only generation columns

In [ ]:
required_columns = ["offensive_text", "counterspeech_text"]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

df = df[required_columns].copy()

df.head()

,offensive_text,counterspeech_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",মেয়েদের বুদ্ধি নিয়ে হেয় করা যুক্তিসঙ্গত নয়; অন...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ","ভাই, মত না মিললেও নারীর ব্যক্তিগত জীবন নিয়ে সম..."
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,"দেখেন, হিন্দুদের সম্পর্কে ধারণা নিয়ে চাপ দিলে ..."
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,"ভাই, মেয়েদের গাড়ি চালানো নিয়ে অসম্মান নয়, নিজে..."
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,কারও বুদ্ধি টেনে অপমান না করে বক্তব্য দিন; সম্...


# 6: Clean text

In [ ]:
def clean_text(text):
    text = str(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("\u200c", "")
    text = text.replace("\u200d", "")
    return text

df["offensive_text"] = df["offensive_text"].apply(clean_text)
df["counterspeech_text"] = df["counterspeech_text"].apply(clean_text)

df = df.dropna()
df = df.drop_duplicates()
df = df[df["offensive_text"].str.len() > 2]
df = df[df["counterspeech_text"].str.len() > 2]

print("Cleaned dataset shape:", df.shape)
df.head()

Cleaned dataset shape: (3011, 2)


,offensive_text,counterspeech_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",মেয়েদের বুদ্ধি নিয়ে হেয় করা যুক্তিসঙ্গত নয়; অন...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ","ভাই, মত না মিললেও নারীর ব্যক্তিগত জীবন নিয়ে সম..."
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,"দেখেন, হিন্দুদের সম্পর্কে ধারণা নিয়ে চাপ দিলে ..."
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,"ভাই, মেয়েদের গাড়ি চালানো নিয়ে অসম্মান নয়, নিজে..."
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,কারও বুদ্ধি টেনে অপমান না করে বক্তব্য দিন; সম্...


#7: Create input and target text

In [ ]:
TASK_PREFIX = "ভদ্র নিরাপদ বাংলা প্রতিউত্তর: "

df["input_text"] = TASK_PREFIX + df["offensive_text"]
df["target_text"] = df["counterspeech_text"]

df = df[["input_text", "target_text"]]

df.head()

,input_text,target_text
0,"ভদ্র নিরাপদ বাংলা প্রতিউত্তর: কি মনে করছ, মেয়ে...",মেয়েদের বুদ্ধি নিয়ে হেয় করা যুক্তিসঙ্গত নয়; অন...
1,"ভদ্র নিরাপদ বাংলা প্রতিউত্তর: তুই তো বেশ্যা, র...","ভাই, মত না মিললেও নারীর ব্যক্তিগত জীবন নিয়ে সম..."
2,ভদ্র নিরাপদ বাংলা প্রতিউত্তর: হিন্দুরাই দেশের ...,"দেখেন, হিন্দুদের সম্পর্কে ধারণা নিয়ে চাপ দিলে ..."
3,ভদ্র নিরাপদ বাংলা প্রতিউত্তর: মাইয়া মানুষ আবার...,"ভাই, মেয়েদের গাড়ি চালানো নিয়ে অসম্মান নয়, নিজে..."
4,ভদ্র নিরাপদ বাংলা প্রতিউত্তর: এই ছাগলের কথা শু...,কারও বুদ্ধি টেনে অপমান না করে বক্তব্য দিন; সম্...


# 8: Train-validation-test split

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

print("Train size:", len(train_df))
print("Validation size:", len(valid_df))
print("Test size:", len(test_df))


Train size: 2408
Validation size: 301
Test size: 302


# 9: Convert to Hugging Face Dataset

In [ ]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(valid_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True))
})

dataset

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 2408
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 301
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 302
    })
})

# 10: Load mBART50

In [ ]:
SRC_LANG = "bn_IN"
TGT_LANG = "bn_IN"

In [ ]:
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

SRC_LANG = "bn_IN"
TGT_LANG = "bn_IN"

tokenizer.src_lang = SRC_LANG
tokenizer.tgt_lang = TGT_LANG

model.config.use_cache = False
model.gradient_checkpointing_enable()

# Bengali language IDs
bn_token_id = tokenizer.lang_code_to_id[TGT_LANG]

# Keep decoder_start_token_id in config if needed
if model.config.decoder_start_token_id is None:
    model.config.decoder_start_token_id = bn_token_id

# IMPORTANT FIX:
# Do NOT set model.config.forced_bos_token_id
# Put generation parameters in generation_config instead
model.generation_config.forced_bos_token_id = bn_token_id
model.generation_config.decoder_start_token_id = model.config.decoder_start_token_id

model.to(device)

print("Model:", MODEL_NAME)
print("Source language:", SRC_LANG)
print("Target language:", TGT_LANG)
print("Bengali token id:", bn_token_id)
print("Pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("EOS token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("Decoder start token id:", model.config.decoder_start_token_id)
print("Generation forced BOS token id:", model.generation_config.forced_bos_token_id)

config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.44GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Model: facebook/mbart-large-50-many-to-many-mmt
Source language: bn_IN
Target language: bn_IN
Bengali token id: 250028
Pad token: <pad> 1
EOS token: </s> 2
Decoder start token id: 2
Generation forced BOS token id: 250028


# 11: Tokenize dataset

In [ ]:
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 96

def preprocess_function(examples):
    tokenizer.src_lang = SRC_LANG
    tokenizer.tgt_lang = TGT_LANG

    inputs = examples["input_text"]
    targets = examples["target_text"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        padding="max_length",
        truncation=True
    )

    target_tokens = tokenizer(
        targets,
        max_length=MAX_TARGET_LENGTH,
        padding="max_length",
        truncation=True
    )

    labels = target_tokens["input_ids"]

    labels = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels
    ]

    model_inputs["labels"] = labels

    return model_inputs


tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["input_text", "target_text"]
)

tokenized_dataset

Map:   0%|          | 0/2408 [00:00<?, ? examples/s]

Map:   0%|          | 0/301 [00:00<?, ? examples/s]

Map:   0%|          | 0/302 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2408
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 301
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 302
    })
})

# 12: Debug tokenization

In [ ]:
sample = tokenized_dataset["train"][0]

decoded_input = tokenizer.decode(
    [x for x in sample["input_ids"] if x != tokenizer.pad_token_id],
    skip_special_tokens=True
)

decoded_label = tokenizer.decode(
    [x for x in sample["labels"] if x != -100],
    skip_special_tokens=True
)

print("Decoded input:")
print(decoded_input)

print("\nDecoded label:")
print(decoded_label)

valid_label_tokens = [x for x in sample["labels"] if x != -100]
print("\nValid label token count:", len(valid_label_tokens))

Decoded input:
ভদ্র নিরাপদ বাংলা প্রতিউত্তর: নারীরা এগিয়ে যাচ্ছে, কিন্তু কোথায় সেটা বললে হুরেরা এসে আমার চৌদ্দগুষ্টি তুলে গালাগালি করবে।

Decoded label:
মেয়েদের যোগ্যতা টেনে অপমান না করে বক্তব্য দিন; ভদ্রভাবে বলুন।

Valid label token count: 23


# 13: Data collator

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)

# 14: Batch loss test

In [ ]:
from torch.utils.data import DataLoader

small_train_dataset = tokenized_dataset["train"].select(range(2))

test_loader = DataLoader(
    small_train_dataset,
    batch_size=1,
    collate_fn=data_collator
)

batch = next(iter(test_loader))
batch = {k: v.to(device) for k, v in batch.items()}

print("Batch input_ids shape:", batch["input_ids"].shape)
print("Batch labels shape:", batch["labels"].shape)

non_ignored_labels = (batch["labels"] != -100).sum().item()
print("Non-ignored label tokens:", non_ignored_labels)

with torch.no_grad():
    outputs = model(**batch)

print("Test batch loss:", outputs.loss.item())

Batch input_ids shape: torch.Size([1, 128])
Batch labels shape: torch.Size([1, 96])
Non-ignored label tokens: 23
Test batch loss: 9.903650283813477


# 15: Training arguments

In [ ]:
OUTPUT_DIR = "/content/mbart50_counterspeech_model"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-5,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=16,

    num_train_epochs=8,

    weight_decay=0.01,

    predict_with_generate=False,

    fp16=torch.cuda.is_available(),

    logging_steps=25,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",

    seed=SEED
)

# 16: Create trainer

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,
    data_collator=data_collator,

    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3)
    ]
)

# 17: Train mBART50

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,39.885920,2.197387
2,26.681233,1.495513
3,21.417913,1.210274
4,18.146298,1.082723
5,16.543004,1.009248
6,15.898468,0.971823
7,15.406512,0.946669
8,14.682140,0.939862


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1208, training_loss=24.153087565441005, metrics={'train_runtime': 7201.4728, 'train_samples_per_second': 2.675, 'train_steps_per_second': 0.168, 'total_flos': 5218448481189888.0, 'train_loss': 24.153087565441005, 'epoch': 8.0})

# 18: Evaluate loss

In [ ]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch
14.682140,0.939862,8


{'eval_loss': 0.9398615956306458}

# 19: Save model

In [ ]:
FINAL_MODEL_DIR = "/content/final_mbart50_counterspeech"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Model saved to:", FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/final_mbart50_counterspeech


# 20: Manual generation test

In [ ]:
model.eval()

def generate_counterspeech(text):
    tokenizer.src_lang = SRC_LANG

    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,

            forced_bos_token_id=tokenizer.lang_code_to_id[TGT_LANG],

            max_new_tokens=70,
            min_new_tokens=6,

            num_beams=5,
            no_repeat_ngram_size=2,
            repetition_penalty=1.15,
            length_penalty=0.9,

            early_stopping=True
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


for i in range(5):
    text = test_df["input_text"].iloc[i]
    ref = test_df["target_text"].iloc[i]
    pred = generate_counterspeech(text)

    print("Input:", text)
    print("Reference:", ref)
    print("Generated:", pred)
    print("Generated length:", len(pred.split()))
    print("-" * 80)

[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input: ভদ্র নিরাপদ বাংলা প্রতিউত্তর: কি মনে করছ, মেয়েরা কিছু বোঝে ?
Reference: মেয়েদের বুদ্ধি নিয়ে হেয় করা যুক্তিসঙ্গত নয়; অনুমান এড়িয়ে সুনির্দিষ্ট তথ্য দিন।
Generated: নারী বলে কাউকে ছোট করা গুরুতর আইনি ঝামেলা আনতে পারে; ভাষা সামলান।
Generated length: 12
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input: ভদ্র নিরাপদ বাংলা প্রতিউত্তর: আরে আরে ইবলিশের ছোট ভাই বলিস কি তুই! প্রতারক মূল জিনিস এড়িয়ে যাস!
Reference: একটু যাচাই করে, প্রতারণার অভিযোগ নিয়ে নিশ্চিত হওয়ার আগে উৎসটা দেখুন; প্রমাণটা গুরুত্বপূর্ণ।
Generated: একটু ভেবে, এই কথা নিয়ে রাগের আগে মানুষটার অনুভূতিটা দেখুন; আঘাত কমবে।
Generated length: 12
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input: ভদ্র নিরাপদ বাংলা প্রতিউত্তর: নষ্ট সে নষ্টদের তো সাথে থাকবে, এটাই স্বভাবিক।
Reference: কথাটা সহজ রাখেন, ব্যক্তিগত মন্তব্য নিয়ে ব্যক্তিগত খোঁচা বন্ধ করুন; ভদ্রভাবে দ্বিমত জানান।
Generated: ভাই, এই ভাষা নিয়ে রাগটা বলুন, মানুষকে গালি দেবেন না; কথা পরিষ্কার হবে।
Generated length: 13
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input: ভদ্র নিরাপদ বাংলা প্রতিউত্তর: ভারতীয় মানুষ গুলোই শুকরের মত আচরনের জাতি
Reference: ভারত নিয়ে মানুষকে হেয় করা ন্যায্য আচরণ নয়; কথাও গ্রহণযোগ্য হবে।
Generated: দেখেন, ভারত নিয়ে অসম্মান না করে মতটা বলুন; এটাই ন্যায্য।
Generated length: 10
--------------------------------------------------------------------------------
Input: ভদ্র নিরাপদ বাংলা প্রতিউত্তর: নাস্তিক ব্লগারদের সাথে তর্ক বিতর্ক করে লাভ নাই কারণ তারা গু মুত খায়
Reference: সোজা কথা, গালির বদলে ধর্মীয় পরিচয় নিয়ে কারণটা বলুন; এটাই যথেষ্ট।
Generated: ভাই, এই ভাষা নিয়ে রাগটা বলুন, মানুষকে গালি দেবেন না; কথা পরিষ্কার হবে।
Generated length: 13
--------------------------------------------------------------------------------


# 21: Generate full test predictions}


In [ ]:
test_inputs = test_df["input_text"].tolist()
references = test_df["target_text"].tolist()

predictions = []

for i, text in enumerate(test_inputs):
    pred = generate_counterspeech(text)
    predictions.append(pred)

    if i % 50 == 0:
        print(f"Generated {i}/{len(test_inputs)}")

results_df = pd.DataFrame({
    "input_text": test_inputs,
    "reference_counterspeech": references,
    "generated_counterspeech": predictions
})

results_df["generated_length"] = results_df["generated_counterspeech"].apply(lambda x: len(str(x).split()))

results_df.head(10)

[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated 0/302


[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Generated 50/302


[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

...

[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Generated 250/302


[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=70) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Generated 300/302


,input_text,reference_counterspeech,generated_counterspeech,generated_length
0,"ভদ্র নিরাপদ বাংলা প্রতিউত্তর: কি মনে করছ, মেয়ে...",মেয়েদের বুদ্ধি নিয়ে হেয় করা যুক্তিসঙ্গত নয়; অন...,নারী বলে কাউকে ছোট করা গুরুতর আইনি ঝামেলা আনতে...,12
1,ভদ্র নিরাপদ বাংলা প্রতিউত্তর: আরে আরে ইবলিশের ...,"একটু যাচাই করে, প্রতারণার অভিযোগ নিয়ে নিশ্চিত ...","একটু ভেবে, এই কথা নিয়ে রাগের আগে মানুষটার অনু...",12
2,ভদ্র নিরাপদ বাংলা প্রতিউত্তর: নষ্ট সে নষ্টদের ...,"কথাটা সহজ রাখেন, ব্যক্তিগত মন্তব্য নিয়ে ব্যক্ত...","ভাই, এই ভাষা নিয়ে রাগটা বলুন, মানুষকে গালি দে...",13
3,ভদ্র নিরাপদ বাংলা প্রতিউত্তর: ভারতীয় মানুষ গুল...,ভারত নিয়ে মানুষকে হেয় করা ন্যায্য আচরণ নয়; কথা...,"দেখেন, ভারত নিয়ে অসম্মান না করে মতটা বলুন; এট...",10
4,ভদ্র নিরাপদ বাংলা প্রতিউত্তর: নাস্তিক ব্লগারদে...,"সোজা কথা, গালির বদলে ধর্মীয় পরিচয় নিয়ে কারণটা ...","ভাই, এই ভাষা নিয়ে রাগটা বলুন, মানুষকে গালি দে...",13
5,ভদ্র নিরাপদ বাংলা প্রতিউত্তর: মোবাইলগুলো কেড়ে ...,"নিজের মানুষকে ভাবুন, নারীরা এই কথায় কষ্ট পেতে ...","ভাই, কারও পরিবার নিয়ে কটু কথা বাদ দিয়ে কারণট...",13
6,ভদ্র নিরাপদ বাংলা প্রতিউত্তর: ওইটা আমার বাপ আর...,"ভাই, আপনার অনুভূতি বুঝি, বাপ-মা নিয়ে শান্তভাবে...","একটা কথা, বাবা নিয়ে রাগটা বলুন, মানুষকে গালি ...",13
7,ভদ্র নিরাপদ বাংলা প্রতিউত্তর: এই মাদকাসক্ত পাট...,"সত্যি বলতে, বুদ্ধি বা যোগ্যতা নিয়ে বিরোধ থাকুক...","ভাই, কারও পরিবারকে গালি না দিয়ে কথার কারণটা ব...",12
8,ভদ্র নিরাপদ বাংলা প্রতিউত্তর: এসব জানোয়ারদের ম...,"মত আলাদা হলেও, অপমান না করেও মত বলা যায়; সম্মা...","দেখেন, কারও পরিবার নিয়ে কটু কথা বাদ দিয়ে কার...",13
9,ভদ্র নিরাপদ বাংলা প্রতিউত্তর: তুই একটা পাঠান স...,"সত্যি বলতে, এই মন্তব্য নিয়ে রাগ থাকলেও গালিটা ...","একটু ভেবে, এই কথা নিয়ে রাগের আগে মানুষটার অনু...",12


# 22: Check generated length

In [ ]:
print(results_df["generated_length"].describe())

results_df[["reference_counterspeech", "generated_counterspeech", "generated_length"]].head(20)

count    302.000000
mean      11.996689
std        1.042290
min        9.000000
25%       11.000000
50%       12.000000
75%       13.000000
max       14.000000
Name: generated_length, dtype: float64


,reference_counterspeech,generated_counterspeech,generated_length
0,মেয়েদের বুদ্ধি নিয়ে হেয় করা যুক্তিসঙ্গত নয়; অন...,নারী বলে কাউকে ছোট করা গুরুতর আইনি ঝামেলা আনতে...,12
1,"একটু যাচাই করে, প্রতারণার অভিযোগ নিয়ে নিশ্চিত ...","একটু ভেবে, এই কথা নিয়ে রাগের আগে মানুষটার অনু...",12
2,"কথাটা সহজ রাখেন, ব্যক্তিগত মন্তব্য নিয়ে ব্যক্ত...","ভাই, এই ভাষা নিয়ে রাগটা বলুন, মানুষকে গালি দে...",13
3,ভারত নিয়ে মানুষকে হেয় করা ন্যায্য আচরণ নয়; কথা...,"দেখেন, ভারত নিয়ে অসম্মান না করে মতটা বলুন; এট...",10
4,"সোজা কথা, গালির বদলে ধর্মীয় পরিচয় নিয়ে কারণটা ...","ভাই, এই ভাষা নিয়ে রাগটা বলুন, মানুষকে গালি দে...",13
5,"নিজের মানুষকে ভাবুন, নারীরা এই কথায় কষ্ট পেতে ...","ভাই, কারও পরিবার নিয়ে কটু কথা বাদ দিয়ে কারণট...",13
6,"ভাই, আপনার অনুভূতি বুঝি, বাপ-মা নিয়ে শান্তভাবে...","একটা কথা, বাবা নিয়ে রাগটা বলুন, মানুষকে গালি ...",13
7,"সত্যি বলতে, বুদ্ধি বা যোগ্যতা নিয়ে বিরোধ থাকুক...","ভাই, কারও পরিবারকে গালি না দিয়ে কথার কারণটা ব...",12
8,"মত আলাদা হলেও, অপমান না করেও মত বলা যায়; সম্মা...","দেখেন, কারও পরিবার নিয়ে কটু কথা বাদ দিয়ে কার...",13
9,"সত্যি বলতে, এই মন্তব্য নিয়ে রাগ থাকলেও গালিটা ...","একটু ভেবে, এই কথা নিয়ে রাগের আগে মানুষটার অনু...",12


# 23: Save predictions

In [ ]:
prediction_csv = "/content/mbart50_test_predictions.csv"

results_df.to_csv(prediction_csv, index=False, encoding="utf-8-sig")

print("Saved predictions:", prediction_csv)

files.download(prediction_csv)

Saved predictions: /content/mbart50_test_predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 24: Metric helper functions

In [ ]:
smooth = SmoothingFunction().method1

def tokenize_bn(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.split()


def calculate_bleu_scores(predictions, references):
    bleu_1_scores = []
    bleu_2_scores = []
    bleu_3_scores = []
    bleu_4_scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = tokenize_bn(pred)
        ref_tokens = tokenize_bn(ref)

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            bleu_1_scores.append(0)
            bleu_2_scores.append(0)
            bleu_3_scores.append(0)
            bleu_4_scores.append(0)
            continue

        bleu_1_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(1, 0, 0, 0),
            smoothing_function=smooth
        ))

        bleu_2_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(0.5, 0.5, 0, 0),
            smoothing_function=smooth
        ))

        bleu_3_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(1/3, 1/3, 1/3, 0),
            smoothing_function=smooth
        ))

        bleu_4_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smooth
        ))

    return {
        "BLEU-1": float(np.mean(bleu_1_scores)),
        "BLEU-2": float(np.mean(bleu_2_scores)),
        "BLEU-3": float(np.mean(bleu_3_scores)),
        "BLEU-4": float(np.mean(bleu_4_scores))
    }


def calculate_meteor(predictions, references):
    scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = tokenize_bn(pred)
        ref_tokens = tokenize_bn(ref)

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            scores.append(0)
            continue

        try:
            score = meteor_score([ref_tokens], pred_tokens)
        except:
            score = 0

        scores.append(score)

    return float(np.mean(scores))


def get_ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)]))


def calculate_diversity(predictions):
    all_unigrams = []
    all_bigrams = []

    for text in predictions:
        tokens = tokenize_bn(text)
        all_unigrams.extend(tokens)
        all_bigrams.extend(get_ngrams(tokens, 2))

    distinct_1 = len(set(all_unigrams)) / len(all_unigrams) if len(all_unigrams) > 0 else 0
    distinct_2 = len(set(all_bigrams)) / len(all_bigrams) if len(all_bigrams) > 0 else 0

    diversity = (distinct_1 + distinct_2) / 2

    return {
        "Distinct-1": distinct_1,
        "Distinct-2": distinct_2,
        "Diversity": diversity
    }


def normalize_bn_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def token_jaccard_similarity(a, b):
    a_tokens = set(tokenize_bn(normalize_bn_text(a)))
    b_tokens = set(tokenize_bn(normalize_bn_text(b)))

    if len(a_tokens) == 0 or len(b_tokens) == 0:
        return 0.0

    intersection = len(a_tokens.intersection(b_tokens))
    union = len(a_tokens.union(b_tokens))

    return intersection / union if union > 0 else 0.0


def calculate_novelty(predictions, train_targets, similarity_threshold=0.70):
    train_targets = [normalize_bn_text(x) for x in train_targets]
    predictions = [normalize_bn_text(x) for x in predictions]

    max_similarities = []
    near_copy_count = 0

    for pred in predictions:
        if len(pred.strip()) == 0:
            max_similarities.append(0.0)
            continue

        similarities = [
            token_jaccard_similarity(pred, train_text)
            for train_text in train_targets
        ]

        max_similarity = max(similarities)
        max_similarities.append(max_similarity)

        if max_similarity >= similarity_threshold:
            near_copy_count += 1

    average_max_similarity = float(np.mean(max_similarities)) if len(max_similarities) > 0 else 0.0
    near_copy_rate = near_copy_count / len(predictions) if len(predictions) > 0 else 0.0

    novelty = 1 - average_max_similarity

    return {
        "Novelty": novelty,
        "Average Max Train Similarity": average_max_similarity,
        "Near Copy Rate": near_copy_rate
    }


bangla_abuse_words = [
    "চুদ",
    "মাগী",
    "হারামি",
    "শালা",
    "কুত্তা",
    "বাল",
    "খানকি",
    "বেশ্যা",
    "মাদারচোদ",
    "বোকাচোদা"
]


def calculate_abuse_word_rate(predictions, abuse_words):
    count = 0

    for text in predictions:
        text = str(text)
        found = any(word in text for word in abuse_words)

        if found:
            count += 1

    abuse_rate = count / len(predictions) if len(predictions) > 0 else 0

    return {
        "Abuse-word rate": abuse_rate,
        "Abusive outputs": count
    }


def calculate_length_stats(predictions):
    lengths = [len(tokenize_bn(text)) for text in predictions]

    empty_count = sum(1 for text in predictions if len(str(text).strip()) == 0)
    empty_rate = empty_count / len(predictions) if len(predictions) > 0 else 0

    return {
        "Average Generated Length": float(np.mean(lengths)),
        "Empty Output Rate": empty_rate
    }

# 25: Bengali ROUGE

In [ ]:
def simple_tokenize_bn(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.split()


def get_ngrams_list(tokens, n):
    if len(tokens) < n:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]


def rouge_n_score(pred, ref, n=1):
    pred_tokens = simple_tokenize_bn(pred)
    ref_tokens = simple_tokenize_bn(ref)

    pred_ngrams = get_ngrams_list(pred_tokens, n)
    ref_ngrams = get_ngrams_list(ref_tokens, n)

    if len(pred_ngrams) == 0 or len(ref_ngrams) == 0:
        return 0.0

    pred_counts = {}
    ref_counts = {}

    for ng in pred_ngrams:
        pred_counts[ng] = pred_counts.get(ng, 0) + 1

    for ng in ref_ngrams:
        ref_counts[ng] = ref_counts.get(ng, 0) + 1

    overlap = 0

    for ng in ref_counts:
        overlap += min(ref_counts[ng], pred_counts.get(ng, 0))

    recall = overlap / len(ref_ngrams)
    precision = overlap / len(pred_ngrams)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def lcs_length(x, y):
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m):
        for j in range(n):
            if x[i] == y[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[m][n]


def rouge_l_score(pred, ref):
    pred_tokens = simple_tokenize_bn(pred)
    ref_tokens = simple_tokenize_bn(ref)

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    lcs = lcs_length(pred_tokens, ref_tokens)

    recall = lcs / len(ref_tokens)
    precision = lcs / len(pred_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def calculate_custom_rouge(predictions, references):
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for pred, ref in zip(predictions, references):
        rouge1_scores.append(rouge_n_score(pred, ref, n=1))
        rouge2_scores.append(rouge_n_score(pred, ref, n=2))
        rougeL_scores.append(rouge_l_score(pred, ref))

    return {
        "rouge1": float(np.mean(rouge1_scores)),
        "rouge2": float(np.mean(rouge2_scores)),
        "rougeL": float(np.mean(rougeL_scores))
    }


rouge_scores = calculate_custom_rouge(predictions, references)

rouge_scores

{'rouge1': 0.07006213023201217,
 'rouge2': 0.012507814244141095,
 'rougeL': 0.06727906245038408}

# 26: BERTScore

In [ ]:
bertscore = evaluate.load("bertscore")

bertscore_result = bertscore.compute(
    predictions=predictions,
    references=references,
    model_type="xlm-roberta-base",
    lang="bn",
    verbose=True
)

bertscore_precision = float(np.mean(bertscore_result["precision"]))
bertscore_recall = float(np.mean(bertscore_result["recall"]))
bertscore_f1 = float(np.mean(bertscore_result["f1"]))

print("BERTScore Precision:", bertscore_precision)
print("BERTScore Recall:", bertscore_recall)
print("BERTScore F1:", bertscore_f1)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

...

calculating scores...
computing bert embedding.


  0%|          | 0/7 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/5 [00:00<?, ?it/s]

done in 7653.14 seconds, 0.04 sentences/sec
BERTScore Precision: 0.8770830104287887
BERTScore Recall: 0.8795182941765185
BERTScore F1: 0.8782810547099208


# 27: Calculate final metrics

In [ ]:
bleu_scores = calculate_bleu_scores(predictions, references)

meteor = calculate_meteor(predictions, references)

diversity_scores = calculate_diversity(predictions)

novelty_scores = calculate_novelty(
    predictions,
    train_df["target_text"].tolist(),
    similarity_threshold=0.70
)

abuse_scores = calculate_abuse_word_rate(
    predictions,
    bangla_abuse_words
)

length_scores = calculate_length_stats(predictions)

final_metrics = {
    "Model": "mBART50",

    "BLEU-1": bleu_scores["BLEU-1"],
    "BLEU-2": bleu_scores["BLEU-2"],
    "BLEU-3": bleu_scores["BLEU-3"],
    "BLEU-4": bleu_scores["BLEU-4"],

    "ROUGE-1": rouge_scores["rouge1"],
    "ROUGE-2": rouge_scores["rouge2"],
    "ROUGE-L": rouge_scores["rougeL"],

    "BERTScore Precision": bertscore_precision,
    "BERTScore Recall": bertscore_recall,
    "BERTScore F1": bertscore_f1,

    "Diversity": diversity_scores["Diversity"],
    "Distinct-1": diversity_scores["Distinct-1"],
    "Distinct-2": diversity_scores["Distinct-2"],

    "Novelty": novelty_scores["Novelty"],
    "Average Max Train Similarity": novelty_scores["Average Max Train Similarity"],
    "Near Copy Rate": novelty_scores["Near Copy Rate"],

    "Abuse-word rate": abuse_scores["Abuse-word rate"],
    "Abusive outputs": abuse_scores["Abusive outputs"],

    "METEOR": meteor,

    "Average Generated Length": length_scores["Average Generated Length"],
    "Empty Output Rate": length_scores["Empty Output Rate"]
}

metrics_df = pd.DataFrame([final_metrics])

metrics_df

,Model,BLEU-1,BLEU-2,BLEU-3,BLEU-4,ROUGE-1,ROUGE-2,ROUGE-L,BERTScore Precision,BERTScore Recall,...,Distinct-1,Distinct-2,Novelty,Average Max Train Similarity,Near Copy Rate,Abuse-word rate,Abusive outputs,METEOR,Average Generated Length,Empty Output Rate
0,mBART50,0.066575,0.028445,0.01891,0.014001,0.070062,0.012508,0.067279,0.877083,0.879518,...,0.04637,0.088226,0.426447,0.573553,0.115894,0.0,0,0.044743,11.996689,0.0


# 28: Save metrics

In [ ]:
metrics_csv = "/content/mbart50_final_metrics.csv"

metrics_df.to_csv(metrics_csv, index=False, encoding="utf-8-sig")

print("Saved metrics:", metrics_csv)

files.download(metrics_csv)

Saved metrics: /content/mbart50_final_metrics.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 29: Zip and download model

In [ ]:
!zip -r /content/final_mbart50_counterspeech.zip /content/final_mbart50_counterspeech

files.download("/content/final_mbart50_counterspeech.zip")

  adding: content/final_mbart50_counterspeech/ (stored 0%)
  adding: content/final_mbart50_counterspeech/training_args.bin (deflated 53%)
  adding: content/final_mbart50_counterspeech/model.safetensors (deflated 7%)
  adding: content/final_mbart50_counterspeech/tokenizer_config.json (deflated 59%)
  adding: content/final_mbart50_counterspeech/config.json (deflated 61%)
  adding: content/final_mbart50_counterspeech/tokenizer.json (deflated 76%)
  adding: content/final_mbart50_counterspeech/generation_config.json (deflated 45%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>